# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all record sets, with their `@id`, title (name), and show their fields with `@id` and data type where possible.

In [ ]:
# List all record sets with IDs and their fields
record_sets = list(meta.record_sets.values())
print("\nRecord Sets:\n===========")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}\n  name: {getattr(rs, 'name', rs.id)}")
    print("  Fields:")
    for field in rs.fields.values():
        dtype = getattr(field, 'data_type', None)
        print(f"    - Field @id: {field.id} | name: {getattr(field, 'name', field.id)} | data_type: {dtype}")
    print("\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we select the main record set (likely the proteomics table) for extraction. Update `main_record_set_id` if needed based on the overview above.

In [ ]:
# If there is more than one record set, adjust accordingly.
record_set_ids = [rs.id for rs in record_sets]

# Prepare dataframes for all record sets
dataframes = {}
for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(recs)
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Skipping {record_set_id}: {e}")

# Use the first available record set for EDA below:
main_record_set_id = record_set_ids[0]
print(f"Fields in {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
The following code block demonstrates filtering and normalization on a numeric field (please select an actual numeric field `@id` as necessary from your data overview above).

In [ ]:
# Replace with actual numeric field @id (pick from printout above, e.g., 'cr:mw' or similar)
# For demonstration, we pick the first detected numeric column.
df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # Fall back: try to coerce a known field (common in proteome datasets: 'MW', 'cr:mw', etc.)
    possible_numeric = [c for c in df.columns if 'mw' in c.lower() or 'abundance' in c.lower()]
    numeric_field_id = possible_numeric[0] if possible_numeric else df.columns[0]  # last resort

print(f"Using numeric field for analysis: {numeric_field_id}")

# Select a threshold for filtering (e.g., molecular weight > 10000)
try:
    threshold = df[numeric_field_id].dropna().quantile(0.75)
except Exception:
    threshold = 10  # fallback
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:0.2f} (top 5):")
print(filtered_df.head())

# Normalize
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (top 5):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
categorical_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field_id = None
for c in categorical_fields:
    if c != numeric_field_id:
        group_field_id = c
        break
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (top 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of selected numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

# If grouping is present, boxplot by category
if group_field_id is not None:
    plt.figure(figsize=(8, 4))
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive data on protein abundance, modification, and sequence information collected from human mast cell extracellular vesicles.
- Using `mlcroissant`, loading and exploring the metadata and records by `@id` is straightforward, allowing transparent and reproducible workflows.
- Initial EDA and visualization highlight the distribution and potential grouping signals of the numeric fields (e.g., molecular weight or abundance).
- For deeper biomedical or proteomics analysis, one may further leverage the detailed fields and cross-reference UniProt or other bioinformatics resources.